# 23N Field Tests #08 · Industrial air, measured from space

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/main/ft08-industrial-air-from-space/ft08_industrial_air_from_space.ipynb)

**Question.** How has nitrogen dioxide (NO₂) over the Vapi, Ankleshwar and Dahej industrial estates in south
Gujarat changed between 2019 and 2025, measured from space? And do the satellite readings move with the ground
monitors near the estates?

**Why it matters.** In the Central Pollution Control Board's *Comprehensive Environmental Assessment of Industrial
Clusters* (December 2009), Ankleshwar ranked 1st and Vapi 2nd of 88 industrial clusters on the Comprehensive
Environmental Pollution Index (CEPI 88.50 and 88.09). Ground monitors in industrial estates are few, and their
records start late and have gaps. A satellite looks at every estate, every day it is clear, with the same
instrument.

**Method.** Copernicus Sentinel-5P (TROPOMI) Level-2 tropospheric NO₂ columns, about 5.5 × 3.5 km per pixel, from
the satellite's early-afternoon overpass. For each month of 2019–2025 we read a fixed number of days, keep pixels
with `qa_value ≥ 0.75` (the producer's recommendation), and average the pixels within 10 km of each estate. Years are
compared on matched months only, so a cloudy monsoon in one year can't fake a trend. The regional level (median of
a ~140 × 240 km box over south Gujarat) is reported alongside, so a region-wide or processing change is visible.
If you add a free OpenAQ API key, the notebook also pulls daily NO₂ from ground monitors within 15 km of each
estate and correlates them with the satellite month by month.

**Data and licences.** Sentinel-5P L2 NO₂ (collection `sentinel-5p-l2-netcdf`) from the Microsoft Planetary Computer
STAC API: Copernicus Sentinel data terms (free, full and open, including commercial use; attribution "Contains
modified Copernicus Sentinel data 2019–2025"). **No account and no key**: the `planetary-computer` package fetches a
free, anonymous, short-lived access token. The ground check is optional and uses the OpenAQ API v3, which needs a
**free OpenAQ account and API key**; each station's own licence is printed with the data. No AI model weights are
used.

**Runtime.** Roughly 15–25 minutes on free Colab or Kaggle CPU, almost all of it reading the satellite files (only
the slice of each orbit over Gujarat is read). `DAYS_PER_MONTH` and `TIME_BUDGET_MIN` cap it.

*Independent study on public data by 23N AI Labs. Vapi, Ankleshwar and Dahej are used only as locations: this says
nothing about any company, estate authority or regulator, and implies no affiliation with any of them. A
satellite column is not a ground concentration; all numbers are estimates.*

## 1 · Setup

In [ ]:
# CELL: deps
# pystac-client finds the satellite files, planetary-computer signs the download links (anonymous, free),
# h5py + fsspec read just the slice of each file we need over HTTP.
import os, sys, subprocess
if os.environ.get("FT08_SMOKE_TEST", "0") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pystac-client", "planetary-computer", "h5py",
                    "fsspec", "aiohttp", "requests", "gradio"], check=False)

In [ ]:
# CELL: config
import os, sys, re, io, json, math, time, textwrap, subprocess, warnings, urllib.request, datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet: it replaces the satellite and ground
# downloads with synthetic pixels and a synthetic ground series. Never post its output.
SMOKE_TEST = os.environ.get("FT08_SMOKE_TEST", "0") == "1"

FT_NUM = "08"
DATA_START, DATA_END = "2019-01-01", "2025-12-31"
BASE_YEAR, END_YEAR = 2019, 2025        # the headline compares these two years (matched months)
LOCKDOWN_MONTHS = [4, 5]                # April-May 2020, India's national lockdown (a known NO2 dip)
WINTER_MONTHS = [11, 12, 1, 2]          # NO2 is highest in the dry winter months over India
MONSOON_MONTHS = [6, 7, 8, 9]           # and lowest (and cloudiest) in the south-west monsoon

# ---- The three estates. ASSUMPTION: each is a 10 km circle around a published point (Wikipedia coordinates:
# Vapi town; Ankleshwar notified industrial area "Anklesvar INA"; Dahej railway station, inside the Dahej
# industrial area). A 10 km circle is about 3 TROPOMI pixels across.
ESTATES = {
    "Vapi":       (20.3667, 72.9000),
    "Ankleshwar": (21.6269, 73.0119),
    "Dahej":      (21.7091, 72.5895),
}
ESTATE_ORDER = ["Vapi", "Ankleshwar", "Dahej"]
RADIUS_KM = 10.0
# Regional reference: a box over south Gujarat and the Gulf of Khambhat (lon_min, lat_min, lon_max, lat_max).
# Its median pixel is the "regional level". It includes Surat and the sea, by design.
STUDY_BOX = (72.20, 20.00, 73.50, 22.20)

# ---- Satellite data
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "sentinel-5p-l2-netcdf"
PRODUCT = "no2"
MODES = ["RPRO", "OFFL"]                # reprocessed and offline products; near-real-time (NRTI) is not used
NO2_VAR = "PRODUCT/nitrogendioxide_tropospheric_column"   # mol m-2
QA_MIN = 0.75                           # producer's recommendation for tropospheric NO2
QA_ROBUST = 0.50                        # robustness variant (lets in partly cloudy pixels)
MOL_M2_TO_E15 = 6.02214e19 / 1e15       # mol m-2 -> 10^15 molecules cm-2 (the usual unit)
DAYS_PER_MONTH = 8                      # days read per month, evenly spaced; 7 years x 12 x 8 = 672 days
TIME_BUDGET_MIN = 16                    # stop reading after this; days are read round-robin across months
WORKERS = 8                             # parallel downloads
MIN_PIX_PER_OVERPASS = 2                # an overpass counts for an estate only with >= 2 good pixels inside
GRID_DEG = 0.05                         # map grid for the charts and video

# ---- Ground monitors (optional): OpenAQ API v3, free key
OPENAQ_URL = "https://api.openaq.org/v3"
GROUND_RADIUS_M = 15000                 # monitors within 15 km of each estate point
GROUND_MIN_DAY_COVERAGE = 0.75          # a day counts if >= 75 % of its hours were measured
GROUND_MIN_DAYS_PER_MONTH = 10

# ---- Documented fact the post leads with (not computed here; source in the README)
CEPI_FACT = ("In the Central Pollution Control Board's 2009 pollution index (CEPI), Ankleshwar ranked 1st and Vapi 2nd "
             "of 88 industrial clusters")
CEPI_SOURCE = "Central Pollution Control Board, Comprehensive Environmental Assessment of Industrial Clusters, Dec 2009"

# ---- Sanity-check bounds (what is already known)
ANNUAL_RANGE_E15 = (0.3, 40.0)          # yearly mean tropospheric NO2 over an estate, 10^15 molec cm-2
MIN_OVERPASSES_PER_YEAR = 40            # per estate per year, after quality filtering
MIN_MATCHED_MONTHS = 6                  # months with data in both BASE_YEAR and END_YEAR
MAX_ABS_CHANGE = 0.70                   # +-70 % in six years over an estate would be extraordinary
MAX_REGIONAL_CHANGE = 0.40              # a whole-region swing this big points to a processing change
LOCKDOWN_MAX_RATIO = 1.10               # Apr-May 2020 above 110 % of Apr-May 2019 contradicts the known dip
SLOPE_CHECK_MIN_CHANGE = 0.10           # only demand trend/end-point agreement when the change is >= 10 %

# ---- Where things live (main)
REPO_REF = "main"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"

# ---- Brand kit
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
DATA_CREDIT = "Contains modified Copernicus Sentinel data 2019-2025, via Microsoft Planetary Computer"
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| SMOKE_TEST:", SMOKE_TEST)
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: synthetic satellite pixels and ground data. These results are NOT real. Do NOT post.")

## 2 · Data: Sentinel-5P NO₂ over south Gujarat, 2019–2025

The Planetary Computer's STAC API lists every Sentinel-5P Level-2 file. One NO₂ file is one orbit, pole to pole,
a few hundred megabytes. We open each file over HTTP and read only the scanlines that cross south Gujarat
(latitude, longitude, `qa_value` and the tropospheric column). Reprocessed (RPRO) files are preferred over offline
(OFFL) ones for the same orbit. Each file's **processor version** is kept, because a change of retrieval version can
shift NO₂ levels and look like a trend (the checks below look for that).

In [ ]:
# CELL: loaders
LAT0, LAT1 = STUDY_BOX[1], STUDY_BOX[3]
LON0, LON1 = STUDY_BOX[0], STUDY_BOX[2]
FNAME_RE = re.compile(r"_(\d{2})_(\d{6})_\d{8}T\d{6}\.nc$")   # ..._<collection>_<processor>_<production time>.nc

def file_version(href):
    m = FNAME_RE.search(href.split("?")[0])
    return (m.group(1), m.group(2)) if m else ("??", "??????")

def read_orbit_h5(h):
    # h: an open h5py.File of an S5P L2 NO2 product. Returns pixels inside STUDY_BOX with qa >= QA_ROBUST.
    P = h["PRODUCT"]
    lat_ds = P["latitude"]
    mid = lat_ds.shape[2] // 2
    latc = lat_ds[0, :, mid]                                        # one column: which scanlines cross the box?
    rows = np.where((latc > LAT0 - 0.8) & (latc < LAT1 + 0.8))[0]
    if rows.size == 0:
        return None
    r0, r1 = int(rows.min()), int(rows.max()) + 1
    lat = lat_ds[0, r0:r1, :].astype(np.float32)
    lon = P["longitude"][0, r0:r1, :].astype(np.float32)
    qa_ds = P["qa_value"]; qa = qa_ds[0, r0:r1, :].astype(np.float32)
    sf = qa_ds.attrs.get("scale_factor", None)
    if sf is not None:
        qa = qa * float(np.ravel(sf)[0])
    elif np.nanmax(qa) > 1.5:                                       # stored as 0..100 without a scale attribute
        qa = qa / 100.0
    no2_ds = h[NO2_VAR]; no2 = no2_ds[0, r0:r1, :].astype(np.float64)
    fill = no2_ds.attrs.get("_FillValue", None)
    bad = ~np.isfinite(no2) | (np.abs(no2) > 1e30)
    if fill is not None:
        bad |= no2 == float(np.ravel(fill)[0])
    sel = (~bad & (lat >= LAT0) & (lat <= LAT1) & (lon >= LON0) & (lon <= LON1) & (qa >= QA_ROBUST - 1e-6))
    if not sel.any():
        return None
    return pd.DataFrame(dict(lat=lat[sel], lon=lon[sel], qa=np.round(qa[sel], 2),
                             no2=(no2[sel] * MOL_M2_TO_E15).astype(np.float32)))

def read_orbit(href, tries=3):
    import h5py, fsspec, planetary_computer as pc
    last = None
    for k in range(tries):
        try:
            fs = fsspec.filesystem("https")
            with fs.open(pc.sign(href), mode="rb", block_size=2 ** 21, cache_type="blockcache") as f:
                with h5py.File(f, "r") as h:
                    return read_orbit_h5(h)
        except Exception as e:
            last = e; time.sleep(2 * (k + 1))
    raise last

def utc_naive(t):
    t = pd.Timestamp(t)
    return t.tz_convert("UTC").tz_localize(None) if t.tzinfo else t

def search_files():
    import pystac_client
    cat = pystac_client.Client.open(STAC_URL)
    rows = []
    for year in range(int(DATA_START[:4]), int(DATA_END[:4]) + 1):
        dt_range = f"{max(DATA_START, f'{year}-01-01')}/{min(DATA_END, f'{year}-12-31')}"
        try:
            s = cat.search(collections=[COLLECTION], bbox=list(STUDY_BOX), datetime=dt_range,
                           query={"s5p:product_name": {"eq": PRODUCT}})
            items = list(s.items())
        except Exception as e:                                       # if the query extension is refused, filter here
            print("  STAC query filter refused, filtering client-side:", type(e).__name__)
            items = list(cat.search(collections=[COLLECTION], bbox=list(STUDY_BOX), datetime=dt_range).items())
        n0 = len(rows)
        for it in items:
            p = it.properties
            if p.get("s5p:product_name") != PRODUCT or p.get("s5p:processing_mode") not in MODES or PRODUCT not in it.assets:
                continue
            href = it.assets[PRODUCT].href
            coll, proc = file_version(href)
            rows.append(dict(id=it.id, time=utc_naive(it.datetime), mode=p.get("s5p:processing_mode"),
                             orbit=p.get("sat:absolute_orbit"), collection=p.get("s5p:collection_identifier", coll),
                             processor=proc, href=href))
        print(f"  {year}: {len(rows) - n0} NO2 files ({', '.join(MODES)})")
    files = pd.DataFrame(rows)
    if files.empty:
        return files
    # one file per orbit: RPRO before OFFL, then the newest collection
    files["mode_rank"] = files["mode"].map({m: i for i, m in enumerate(MODES)})
    files = (files.sort_values(["orbit", "mode_rank", "collection"], ascending=[True, True, False])
             .drop_duplicates("orbit").drop(columns="mode_rank").sort_values("time").reset_index(drop=True))
    files["date"] = files["time"].dt.normalize()
    return files

def pick_days(files):
    # DAYS_PER_MONTH evenly spaced days per month, then a round-robin order (the 1st chosen day of every month,
    # then the 2nd ...), so if TIME_BUDGET_MIN stops the reading early, every month is still covered evenly.
    order = []
    for _, g in files.groupby(files["date"].dt.to_period("M")):
        days = np.sort(g["date"].unique())
        k = min(DAYS_PER_MONTH, len(days))
        chosen = days[np.linspace(0, len(days) - 1, k).round().astype(int)] if k else []
        for rank, d in enumerate(chosen):
            order.append((rank, d))
    order.sort(key=lambda x: (x[0], x[1]))
    day_rank = {d: r for r, d in order}
    sel = files[files["date"].isin(day_rank)].copy()
    sel["rank"] = sel["date"].map(day_rank)
    return sel.sort_values(["rank", "date", "orbit"]).reset_index(drop=True)

def load_real():
    files = search_files()
    print(f"STAC: {len(files)} NO2 orbit files over the study box")
    if files.empty:
        return files, pd.DataFrame(), ["no files found"]
    todo = pick_days(files)
    print(f"Reading {len(todo)} orbit files on {todo['date'].nunique()} days "
          f"(DAYS_PER_MONTH={DAYS_PER_MONTH}, budget {TIME_BUDGET_MIN} min)")
    parts, errors, done, t0 = [], [], [], time.time()
    with ThreadPoolExecutor(WORKERS) as ex:
        it = iter(todo.itertuples(index=False)); running = {}
        def submit_next():
            try:
                r = next(it)
            except StopIteration:
                return False
            running[ex.submit(read_orbit, r.href)] = r; return True
        for _ in range(WORKERS * 2):
            submit_next()
        while running:
            f = next(as_completed(list(running)))
            r = running.pop(f)
            try:
                df = f.result()
                if df is not None and len(df):
                    df["time"] = r.time; df["orbit"] = r.orbit; df["mode"] = r.mode
                    df["collection"] = r.collection; df["processor"] = r.processor
                    parts.append(df)
                done.append(r.orbit)
            except Exception as e:
                errors.append(f"orbit {r.orbit}: {type(e).__name__}: {str(e)[:120]}")
            n = len(done) + len(errors)
            if n % 25 == 0:
                el = time.time() - t0
                print(f"  {n}/{len(todo)} files · {el / 60:.1f} min · {el / max(n, 1):.1f} s per file (parallel) · errors {len(errors)}")
            if (time.time() - t0) / 60 < TIME_BUDGET_MIN:
                submit_next()
    if len(done) < len(todo):
        print(f"TIME_BUDGET_MIN reached: read {len(done)} of {len(todo)} files (round-robin, so months stay balanced)")
    files["read"] = files["orbit"].isin(done)
    pix = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
        columns=["lat", "lon", "qa", "no2", "time", "orbit", "mode", "collection", "processor"])
    return files, pix, errors

def synthetic_data(seed=8):
    # SMOKE TEST ONLY: random pixels over the box with a winter-high/monsoon-low cycle, a plume at each estate,
    # a slow trend, an April-May 2020 dip and cloud gaps in the monsoon. Not real data.
    rng = np.random.default_rng(seed)
    days = pd.date_range(DATA_START, DATA_END, freq="D")
    days = days[(days.day - 1) % 4 == 0]
    trend = {"Vapi": -0.03, "Ankleshwar": 0.02, "Dahej": 0.05}
    peak = {"Vapi": 6.0, "Ankleshwar": 5.0, "Dahej": 4.0}
    parts, frows = [], []
    for k, d in enumerate(days):
        orbit = 10000 + k
        proc = "020400" if d < pd.Timestamp("2022-07-26") else "020600"
        mode = "RPRO" if d < pd.Timestamp("2022-07-26") else "OFFL"
        frows.append(dict(id=f"synthetic-{orbit}", time=d + pd.Timedelta(hours=8), mode=mode, orbit=orbit,
                          collection="03", processor=proc, href="synthetic", date=d, read=True))
        clear = 0.15 if d.month in MONSOON_MONTHS else 0.8
        n = rng.binomial(1400, clear)
        lat = rng.uniform(LAT0, LAT1, n); lon = rng.uniform(LON0, LON1, n)
        season = 1.0 + 0.45 * np.cos(2 * np.pi * (d.dayofyear - 10) / 365.25)
        v = 1.4 * season + rng.gamma(2.0, 0.35, n)
        yrs = d.year - BASE_YEAR + d.dayofyear / 365.25
        for e, (la, lo) in ESTATES.items():
            r2 = ((lat - la) * 111) ** 2 + ((lon - lo) * 111 * math.cos(math.radians(la))) ** 2
            v += peak[e] * season * (1 + trend[e]) ** yrs * np.exp(-r2 / (2 * 9.0 ** 2))
        if d.year == 2020 and d.month in LOCKDOWN_MONTHS:
            v *= 0.7
        v *= rng.lognormal(0, 0.25, n)
        parts.append(pd.DataFrame(dict(lat=lat.astype(np.float32), lon=lon.astype(np.float32),
                                       qa=np.round(rng.choice([0.5, 0.75, 1.0], n, p=[0.15, 0.25, 0.6]), 2),
                                       no2=v.astype(np.float32), time=d + pd.Timedelta(hours=8), orbit=orbit,
                                       mode=mode, collection="03", processor=proc)))
    return pd.DataFrame(frows), pd.concat(parts, ignore_index=True), []

In [ ]:
# CELL: load_data
CACHE = f"ft{FT_NUM}_pixels.parquet"
if SMOKE_TEST:
    FILES, PIX, READ_ERRORS = synthetic_data()
elif os.path.exists(CACHE):
    PIX = pd.read_parquet(CACHE); FILES = pd.read_csv(CACHE.replace(".parquet", "_files.csv"), parse_dates=["time", "date"])
    READ_ERRORS = []
    print("Loaded pixels from the local cache (delete", CACHE, "to download again).")
else:
    FILES, PIX, READ_ERRORS = load_real()
    if len(PIX):
        PIX.to_parquet(CACHE); FILES.to_csv(CACHE.replace(".parquet", "_files.csv"), index=False)

PIX["time"] = pd.to_datetime(PIX["time"])
PIX["day"] = PIX["time"].dt.normalize()
PIX["year"] = PIX["time"].dt.year
PIX["month"] = PIX["time"].dt.month
print(f"{len(PIX):,} pixels (qa >= {QA_ROBUST}) from {PIX['orbit'].nunique()} overpasses, "
      f"{PIX['time'].min():%d %b %Y} -> {PIX['time'].max():%d %b %Y}")
if READ_ERRORS:
    print(f"{len(READ_ERRORS)} read errors (first 3):", READ_ERRORS[:3])
VERSIONS = (PIX.groupby(["year", "mode", "collection", "processor"])["orbit"].nunique()
            .rename("overpasses").reset_index())
print("Processor versions by year (a change of version can shift NO2 levels):")
print(VERSIONS.to_string(index=False))

## 3 · Method: average the pixels over each estate, compare matched months

1. **Quality.** Keep pixels with `qa_value ≥ QA_MIN` (0.75). This removes cloudy and snow/ice scenes; it also
   removes most monsoon days.
2. **Estates.** For every overpass, the mean of the good pixels whose centre is within `RADIUS_KM` of the estate
   point (at least `MIN_PIX_PER_OVERPASS`). Monthly value = mean over that month's overpasses.
3. **Regional level.** For every overpass, the median good pixel in `STUDY_BOX`; monthly mean of those. The
   estate's **enhancement** is its monthly value minus the regional level.
4. **Change 2019 → 2025.** For each estate, the mean of the months that have data in **both** years, then the ratio.
   Matching months stops a cloudier year (fewer monsoon months) from looking like a cleaner one.
5. **Trend.** The Theil–Sen slope of the yearly means (matched to the calendar months seen in every year where
   possible), as % of the 2019 level per year.

In [ ]:
# CELL: aggregate
def haversine_km(lat, lon, la0, lo0):
    p = np.pi / 180
    a = (np.sin((lat - la0) * p / 2) ** 2 + np.cos(la0 * p) * np.cos(lat * p) * np.sin((lon - lo0) * p / 2) ** 2)
    return 12742 * np.arcsin(np.sqrt(a))

def overpass_means(pix, la, lo, radius_km, qa_min):
    d = pix[(pix.qa >= qa_min - 1e-6)]
    d = d[haversine_km(d.lat.values, d.lon.values, la, lo) <= radius_km]
    g = d.groupby("orbit").agg(time=("time", "first"), no2=("no2", "mean"), n=("no2", "size"))
    return g[g.n >= MIN_PIX_PER_OVERPASS]

def monthly(over):
    if over.empty:
        return pd.Series(dtype=float)
    return over.set_index("time")["no2"].resample("MS").mean()

def regional_monthly(pix, qa_min):
    d = pix[pix.qa >= qa_min - 1e-6]
    g = d.groupby("orbit").agg(time=("time", "first"), no2=("no2", "median"))
    return g.set_index("time")["no2"].resample("MS").mean()

MONTH_INDEX = pd.date_range(DATA_START, DATA_END, freq="MS")
def build_tables(qa_min):
    over = {e: overpass_means(PIX, *ESTATES[e], RADIUS_KM, qa_min) for e in ESTATE_ORDER}
    mon = pd.DataFrame({e: monthly(over[e]) for e in ESTATE_ORDER}).reindex(MONTH_INDEX)
    mon["Regional"] = regional_monthly(PIX, qa_min).reindex(MONTH_INDEX)
    return over, mon

OVER, MON = build_tables(QA_MIN)
OVER_R, MON_R = build_tables(QA_ROBUST)
ENH = MON[ESTATE_ORDER].sub(MON["Regional"], axis=0)       # enhancement over the regional level

def matched_change(series, y0, y1):
    s = series.dropna()
    a = s[s.index.year == y0]; b = s[s.index.year == y1]
    months = sorted(set(a.index.month) & set(b.index.month))
    if not months:
        return float("nan"), float("nan"), float("nan"), 0
    va = a[a.index.month.isin(months)].mean(); vb = b[b.index.month.isin(months)].mean()
    return va, vb, (vb / va - 1) if va > 0 else float("nan"), len(months)

def yearly_means(series):
    # yearly mean over the calendar months that have data in at least 5 of the 7 years (a fixed basket)
    s = series.dropna()
    common = [m for m in range(1, 13) if s[s.index.month == m].index.year.nunique() >= 5]
    s = s[s.index.month.isin(common)]
    return s.groupby(s.index.year).mean(), common

def theil_sen_pct(yearly, base):
    x = yearly.index.values.astype(float); y = yearly.values
    if len(x) < 3 or not base or base != base:
        return float("nan")
    sl = [(y[j] - y[i]) / (x[j] - x[i]) for i in range(len(x)) for j in range(i + 1, len(x))]
    return float(np.median(sl)) / base

RES = {}
for e in ESTATE_ORDER + ["Regional"]:
    v0, v1, chg, nm = matched_change(MON[e], BASE_YEAR, END_YEAR)
    _, _, chg_r, _ = matched_change(MON_R[e], BASE_YEAR, END_YEAR)
    ym, basket = yearly_means(MON[e])
    RES[e] = dict(v0=v0, v1=v1, chg=chg, n_months=nm, chg_qa50=chg_r, yearly=ym, basket=basket,
                  slope=theil_sen_pct(ym, ym.get(BASE_YEAR, float("nan"))))
    if e != "Regional":
        _, _, RES[e]["chg_enh"], _ = matched_change(ENH[e] + 0.0, BASE_YEAR, END_YEAR)
        # 2-year averages, leaving out the 2020 lockdown year: (2019, 2021) vs (END_YEAR-1, END_YEAR)
        s = MON[e].dropna()
        a = s[s.index.year.isin([BASE_YEAR, BASE_YEAR + 2])]; b = s[s.index.year.isin([END_YEAR - 1, END_YEAR])]
        mm = sorted(set(a.index.month) & set(b.index.month))
        RES[e]["chg_2y"] = (b[b.index.month.isin(mm)].mean() / a[a.index.month.isin(mm)].mean() - 1) if mm else float("nan")
        RES[e]["n_over"] = OVER[e].assign(y=lambda d: pd.to_datetime(d.time).dt.year).groupby("y").size()
for e in ESTATE_ORDER + ["Regional"]:
    r = RES[e]
    print(f"{e:11s} {BASE_YEAR}: {r['v0']:.2f}  {END_YEAR}: {r['v1']:.2f}  (1e15 molec/cm², {r['n_months']} matched months)"
          f"  change {r['chg']:+.0%}  trend {r['slope']:+.1%}/yr")

## 4 · Ground check (optional): do the satellite and the ground monitors move together?

A satellite measures the NO₂ in the whole column of air above a ~20 km² pixel at about 13:30 local time; a ground
monitor measures the concentration at one inlet, around the clock. They should **rise and fall together** from
month to month, not match in value. This cell uses the OpenAQ API v3 (a free key from your OpenAQ
account at openaq.org, set as the environment variable or Colab secret
`OPENAQ_API_KEY`). Without a key it skips itself, and the post says the ground check was not run.

In [ ]:
# CELL: ground
def openaq_key():
    k = os.environ.get("OPENAQ_API_KEY", "")
    if not k and IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("OPENAQ_API_KEY") or ""
        except Exception:
            k = ""
    return k.strip()

def oaq_get(path, params, key, tries=4):
    import requests
    for k in range(tries):
        r = requests.get(f"{OPENAQ_URL}{path}", params=params, headers={"X-API-Key": key}, timeout=60)
        if r.status_code == 429:
            time.sleep(10 * (k + 1)); continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()

def ground_stations(key):
    rows = []
    for e in ESTATE_ORDER:
        la, lo = ESTATES[e]
        js = oaq_get("/locations", dict(coordinates=f"{la:.4f},{lo:.4f}", radius=GROUND_RADIUS_M, limit=100), key)
        for loc in js.get("results", []):
            for s in loc.get("sensors", []):
                if (s.get("parameter") or {}).get("name", "").lower() == "no2":
                    c = loc.get("coordinates") or {}
                    lic = "; ".join(l.get("name", "") for l in (loc.get("licenses") or []))
                    rows.append(dict(estate=e, location_id=loc["id"], station=loc.get("name"), sensor_id=s["id"],
                                     units=s["parameter"].get("units"), lat=c.get("latitude"), lon=c.get("longitude"),
                                     provider=(loc.get("provider") or {}).get("name"), licence=lic))
        time.sleep(1)
    return pd.DataFrame(rows).drop_duplicates("sensor_id")

def ground_daily(sensor_id, key):
    out, page = [], 1
    while True:
        js = oaq_get(f"/sensors/{sensor_id}/days", dict(date_from=DATA_START, date_to=DATA_END, limit=1000, page=page), key)
        res = js.get("results", [])
        for r in res:
            cov = r.get("coverage") or {}
            exp, obs = cov.get("expectedCount") or 0, cov.get("observedCount") or 0
            t = ((r.get("period") or {}).get("datetimeFrom") or {}).get("utc")
            if t is not None and r.get("value") is not None:
                out.append(dict(time=utc_naive(t), value=float(r["value"]), cov=(obs / exp) if exp else np.nan))
        if len(res) < 1000:
            break
        page += 1; time.sleep(1)
    return pd.DataFrame(out)

def synthetic_ground():
    # SMOKE TEST ONLY: one fake monitor per estate that follows the synthetic satellite series with noise.
    rng = np.random.default_rng(3); rows, mons = [], {}
    for e in ESTATE_ORDER:
        la, lo = ESTATES[e]
        rows.append(dict(estate=e, location_id=0, station=f"synthetic monitor {e}", sensor_id=hash(e) % 10000,
                         units="µg/m³", lat=la + 0.03, lon=lo + 0.02, provider="synthetic", licence="synthetic"))
        s = MON[e].copy(); s = s[s.index >= "2021-06-01"]
        mons[rows[-1]["sensor_id"]] = (s * 4.0 + rng.normal(0, 4, len(s))).clip(lower=1)
    return pd.DataFrame(rows), mons

GROUND_STATUS, STATIONS, GROUND_MON, GROUND_PAIRS = "not run", pd.DataFrame(), {}, pd.DataFrame()
KEY = "" if SMOKE_TEST else openaq_key()
try:
    if SMOKE_TEST and os.environ.get("FT08_SMOKE_GROUND", "1") == "1":
        STATIONS, GROUND_MON = synthetic_ground()
    elif KEY:
        STATIONS = ground_stations(KEY)
        print(f"OpenAQ: {len(STATIONS)} NO2 sensors within {GROUND_RADIUS_M / 1000:.0f} km of the estates")
        for s in STATIONS.itertuples():
            d = ground_daily(s.sensor_id, KEY)
            if d.empty:
                continue
            d = d[d["cov"] >= GROUND_MIN_DAY_COVERAGE].set_index("time")["value"]
            g = d.resample("MS").agg(["mean", "size"])
            GROUND_MON[s.sensor_id] = g.loc[g["size"] >= GROUND_MIN_DAYS_PER_MONTH, "mean"]
    elif not SMOKE_TEST:
        print("No OPENAQ_API_KEY: ground check skipped (the satellite results do not depend on it).")
    if len(STATIONS):
        print(STATIONS[["estate", "station", "units", "provider", "licence"]].to_string(index=False))
    rows = []
    for s in STATIONS.itertuples():
        g = GROUND_MON.get(s.sensor_id)
        if g is None or not len(g) or pd.isna(s.lat) or pd.isna(s.lon):
            continue
        sat = monthly(overpass_means(PIX, s.lat, s.lon, RADIUS_KM, QA_MIN)).reindex(MONTH_INDEX)
        j = pd.DataFrame(dict(sat=sat, ground=g.reindex(MONTH_INDEX))).dropna()
        for t, r in j.iterrows():
            rows.append(dict(estate=s.estate, station=s.station, sensor_id=s.sensor_id, units=s.units, time=t, **r))
    GROUND_PAIRS = pd.DataFrame(rows)
    if len(GROUND_PAIRS):
        GROUND_STATUS = "run"
except Exception as e:
    GROUND_STATUS = f"failed ({type(e).__name__}: {str(e)[:100]})"
    print("Ground check failed:", GROUND_STATUS)

GROUND_R, GROUND_N, GROUND_BY_STATION = float("nan"), 0, pd.DataFrame()
if GROUND_STATUS == "run":
    # correlation of monthly anomalies within each station (so different station levels don't inflate r)
    gp = GROUND_PAIRS.copy()
    for c in ("sat", "ground"):
        gp[c + "_z"] = gp.groupby("sensor_id")[c].transform(lambda x: (x - x.mean()) / (x.std() or 1))
    GROUND_R, GROUND_N = float(gp[["sat_z", "ground_z"]].corr().iloc[0, 1]), len(gp)
    GROUND_BY_STATION = (gp.groupby(["estate", "station"]).apply(lambda d: pd.Series(
        dict(months=len(d), r=d["sat"].corr(d["ground"]) if len(d) > 2 else np.nan)), include_groups=False).reset_index())
    print(GROUND_BY_STATION.round(2).to_string(index=False))
    print(f"Satellite vs ground, monthly, all stations: r = {GROUND_R:.2f} over {GROUND_N} station-months")

## 5 · Results: the headline, after the sanity checks

In [ ]:
# CELL: results
PROBLEMS, NOTES = [], []
def pct(v):
    return f"{v:+.0%}" if v == v else "n/a"

# coverage
for e in ESTATE_ORDER:
    for y in range(BASE_YEAR, END_YEAR + 1):
        n = int(RES[e]["n_over"].get(y, 0))
        if n < MIN_OVERPASSES_PER_YEAR:
            PROBLEMS.append(f"{e}: only {n} usable overpasses in {y} (< {MIN_OVERPASSES_PER_YEAR}): download gap or too few days read")
    if RES[e]["n_months"] < MIN_MATCHED_MONTHS:
        PROBLEMS.append(f"{e}: only {RES[e]['n_months']} months with data in both {BASE_YEAR} and {END_YEAR} (< {MIN_MATCHED_MONTHS})")
# plausible levels and changes
for e in ESTATE_ORDER:
    for y, v in RES[e]["yearly"].items():
        if not (ANNUAL_RANGE_E15[0] <= v <= ANNUAL_RANGE_E15[1]):
            PROBLEMS.append(f"{e} {y}: yearly mean {v:.2f} x10^15 molec/cm² outside {ANNUAL_RANGE_E15}: unit or reading error")
    c = RES[e]["chg"]
    if c == c and abs(c) > MAX_ABS_CHANGE:
        PROBLEMS.append(f"{e}: change {c:+.0%} beyond ±{MAX_ABS_CHANGE:.0%}: implausible, check versions and coverage")
    s = RES[e]["slope"]
    if c == c and s == s and abs(c) >= SLOPE_CHECK_MIN_CHANGE and np.sign(c) != np.sign(s):
        PROBLEMS.append(f"{e}: {BASE_YEAR}->{END_YEAR} change {c:+.0%} but the 7-year trend is {s:+.1%}/yr: an end-point artefact, not a trend")
    if c == c and RES[e]["chg_2y"] == RES[e]["chg_2y"] and abs(c) >= SLOPE_CHECK_MIN_CHANGE and np.sign(c) != np.sign(RES[e]["chg_2y"]):
        PROBLEMS.append(f"{e}: sign flips with 2-year averages ({c:+.0%} vs {RES[e]['chg_2y']:+.0%})")
    for k, lab in (("chg_qa50", f"qa >= {QA_ROBUST}"), ("chg_enh", "enhancement over the region")):
        v = RES[e][k]
        if c == c and v == v and np.sign(c) != np.sign(v) and abs(c) >= 0.05:
            NOTES.append(f"robustness: {e} change flips sign with {lab} ({c:+.0%} vs {v:+.0%}): say the change is not robust")
rc = RES["Regional"]["chg"]
if rc == rc and abs(rc) > MAX_REGIONAL_CHANGE:
    PROBLEMS.append(f"regional level changed {rc:+.0%} ({BASE_YEAR}->{END_YEAR}): a region-wide swing this big suggests a processing change")
# known facts: the lockdown dip and the seasonal cycle
est = MON[ESTATE_ORDER].mean(axis=1)
ld19 = est[(est.index.year == 2019) & est.index.month.isin(LOCKDOWN_MONTHS)].mean()
ld20 = est[(est.index.year == 2020) & est.index.month.isin(LOCKDOWN_MONTHS)].mean()
LOCKDOWN_RATIO = ld20 / ld19 if ld19 and ld19 == ld19 else float("nan")
if LOCKDOWN_RATIO == LOCKDOWN_RATIO and LOCKDOWN_RATIO > LOCKDOWN_MAX_RATIO:
    PROBLEMS.append(f"Apr-May 2020 (lockdown) is {LOCKDOWN_RATIO:.0%} of Apr-May 2019: contradicts the documented lockdown dip")
elif LOCKDOWN_RATIO != LOCKDOWN_RATIO:
    NOTES.append("no Apr-May data for 2019 or 2020: lockdown check skipped")
WINTER_MEAN = est[est.index.month.isin(WINTER_MONTHS)].mean()
MONSOON_MEAN = est[est.index.month.isin(MONSOON_MONTHS)].mean()
if WINTER_MEAN == WINTER_MEAN and MONSOON_MEAN == MONSOON_MEAN and WINTER_MEAN <= MONSOON_MEAN:
    PROBLEMS.append(f"winter mean ({WINTER_MEAN:.2f}) is not above the monsoon mean ({MONSOON_MEAN:.2f}): contradicts the known seasonal cycle")
# processor versions: a version break can look like a trend
def major(p):
    return p[:2] if isinstance(p, str) and len(p) == 6 else "??"
V_BASE = sorted(VERSIONS[VERSIONS.year == BASE_YEAR].processor.astype(str).unique())
V_END = sorted(VERSIONS[VERSIONS.year == END_YEAR].processor.astype(str).unique())
if {major(p) for p in V_BASE} != {major(p) for p in V_END}:
    PROBLEMS.append(f"processor major version differs: {BASE_YEAR} {V_BASE} vs {END_YEAR} {V_END}. Check the release notes "
                    "for a change in tropospheric columns before comparing these years")
elif V_BASE != V_END:
    NOTES.append(f"processor versions: {BASE_YEAR} {V_BASE}, {END_YEAR} {V_END} (same major version; check the release notes)")
if (VERSIONS.processor.astype(str).apply(major) == "01").any():
    PROBLEMS.append("some files are processor 01.xx (pre-2021, biased low and not reprocessed): use RPRO/collection 03 only")
# ground
if GROUND_STATUS == "run":
    if GROUND_R < 0:
        PROBLEMS.append(f"satellite and ground monitors move in opposite directions (r = {GROUND_R:.2f}): check the stations")
    elif GROUND_R < 0.3:
        NOTES.append(f"satellite vs ground correlation is weak (r = {GROUND_R:.2f}): say so in the post")
else:
    NOTES.append(f"ground check {GROUND_STATUS}: the post must not claim a ground comparison")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE TEST: synthetic satellite pixels")

print("Assumptions:", dict(RADIUS_KM=RADIUS_KM, QA_MIN=QA_MIN, STUDY_BOX=STUDY_BOX, DAYS_PER_MONTH=DAYS_PER_MONTH,
      MODES=MODES, BASE_YEAR=BASE_YEAR, END_YEAR=END_YEAR, ESTATES=ESTATES))
for n in NOTES:
    print("note:", n)
if PROBLEMS:
    print("\n" + "!" * 78)
    for p in PROBLEMS:
        print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers until every failure above is explained or fixed.")
    print("!" * 78 + "\n")
else:
    print("All sanity checks passed.")

CHG = {e: RES[e]["chg"] for e in ESTATE_ORDER}
TOP = max(ESTATE_ORDER, key=lambda e: abs(CHG[e]) if CHG[e] == CHG[e] else -1)
def up_down(v):
    return "up" if v > 0 else "down"
HEADLINE = (f"From {BASE_YEAR} to {END_YEAR}, tropospheric NO₂ measured by Sentinel-5P changed by "
            + ", ".join(f"{pct(CHG[e])} over {e}" for e in ESTATE_ORDER)
            + f" (matched months, estimate); the south Gujarat regional level changed {pct(rc)}.")
BIG = pct(CHG[TOP])
BIG_LINE = (f"NO₂ over {TOP}, {BASE_YEAR} to {END_YEAR}. "
            + ", ".join(f"{e} {pct(CHG[e])}" for e in ESTATE_ORDER if e != TOP) + f"; region {pct(rc)}.")
GROUND_LINE = (f"Ground monitors near the estates move with the satellite: r = {GROUND_R:.2f} over {GROUND_N} station-months."
               if GROUND_STATUS == "run" else "Ground check not run (needs a free OpenAQ key).")
print("\nHEADLINE (estimate):", HEADLINE)
print("  ", GROUND_LINE)
print(f"  units: 10^15 molecules/cm² (tropospheric column). Lockdown Apr-May 2020 vs 2019: {LOCKDOWN_RATIO:.0%}. "
      f"Winter {WINTER_MEAN:.2f} vs monsoon {MONSOON_MEAN:.2f}.")
print("\nRobustness:")
for e in ESTATE_ORDER:
    r = RES[e]
    print(f"  {e:11s} 7-yr trend {r['slope']:+.1%}/yr · 2-yr averages {pct(r['chg_2y'])} · qa>={QA_ROBUST} {pct(r['chg_qa50'])}"
          f" · enhancement over region {pct(r['chg_enh'])}")

In [ ]:
# CELL: results_chart
SERIES_COL = dict(zip(ESTATE_ORDER, [B.MOSS, B.AMBER, B.INK]))
SERIES_COL["Regional"] = B.STONE

def draw_monthly(ax, upto=None, big=False, estates=ESTATE_ORDER):
    m = MON if upto is None else MON[MON.index <= upto]
    for y in range(int(DATA_START[:4]), int(DATA_END[:4]) + 1):
        ax.axvspan(pd.Timestamp(y, MONSOON_MONTHS[0], 1), pd.Timestamp(y, MONSOON_MONTHS[-1], 30), color=B.PAPER_DEEP, lw=0, zorder=0)
    for e in list(estates) + ["Regional"]:
        s = m[e].dropna()
        ax.plot(s.index, s.values, color=SERIES_COL[e], lw=(2.6 if big else 1.8) if e != "Regional" else 1.4,
                ls="-" if e != "Regional" else "--", marker="o", ms=3 if big else 2, label=e if e != "Regional" else "Region (median)")
    ax.set_xlim(pd.Timestamp(DATA_START), pd.Timestamp(DATA_END))
    top = np.nanmax(MON[list(estates) + ["Regional"]].values) if MON.notna().any().any() else 1
    ax.set_ylim(0, top * 1.15)
    ax.set_ylabel("NO₂ column, 10¹⁵ molec/cm²", fontsize=14 if big else 11)
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(matplotlib.dates.YearLocator())
    ax.tick_params(labelsize=13 if big else 10)
    ax.legend(loc="upper left", ncol=4, fontsize=12 if big else 9)

GRID_LAT = np.arange(LAT0, LAT1 + 1e-9, GRID_DEG); GRID_LON = np.arange(LON0, LON1 + 1e-9, GRID_DEG)
def year_grid(year):
    d = PIX[(PIX.year == year) & (PIX.qa >= QA_MIN - 1e-6)]
    s, _, _ = np.histogram2d(d.lat, d.lon, bins=[GRID_LAT, GRID_LON], weights=d.no2)
    n, _, _ = np.histogram2d(d.lat, d.lon, bins=[GRID_LAT, GRID_LON])
    with np.errstate(invalid="ignore", divide="ignore"):
        g = np.where(n >= 3, s / np.maximum(n, 1), np.nan)
    return g
YEARS = list(range(int(DATA_START[:4]), int(DATA_END[:4]) + 1))
GRIDS = {y: year_grid(y) for y in YEARS}
_allv = np.concatenate([g[np.isfinite(g)] for g in GRIDS.values()]) if any(np.isfinite(g).any() for g in GRIDS.values()) else np.array([0, 1])
MAP_VMIN, MAP_VMAX = float(np.percentile(_allv, 2)), float(np.percentile(_allv, 99))

def draw_map(ax, year, big=False, labels=True):
    ax.imshow(GRIDS[year], origin="lower", extent=[LON0, LON1, LAT0, LAT1], cmap=B.SEQ_CMAP,
              vmin=MAP_VMIN, vmax=MAP_VMAX, interpolation="nearest",
              aspect=1 / math.cos(math.radians((LAT0 + LAT1) / 2)))       # true shape: 1° lon is shorter than 1° lat
    th = np.linspace(0, 2 * np.pi, 90)
    for e in ESTATE_ORDER:
        la, lo = ESTATES[e]
        ax.plot(lo + RADIUS_KM / (111 * math.cos(math.radians(la))) * np.cos(th), la + RADIUS_KM / 111 * np.sin(th),
                color=B.AMBER, lw=2.2 if big else 1.5)
        if labels:
            ax.text(lo + 0.13, la, e.upper(), fontproperties=B.FP.get("mono_bold"), fontsize=13 if big else 9,
                    color=B.INK, va="center", bbox=dict(boxstyle="round,pad=0.2", fc=B.PAPER, ec="none", alpha=0.8))
    ax.set_xlim(LON0, LON1); ax.set_ylim(LAT0, LAT1)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

TAG = " · SMOKE TEST, NOT REAL" if SMOKE_TEST else ""
fig, ax = plt.subplots(figsize=(11, 4.8))
draw_monthly(ax)
B.chart_title(ax, "NO₂ over three industrial estates, month by month",
              f"Sentinel-5P tropospheric column · pixels within {RADIUS_KM:.0f} km · qa ≥ {QA_MIN} · shaded: monsoon")
B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT}{TAG}")
fig.tight_layout(); fig.savefig(f"ft{FT_NUM}_monthly.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 7.2))
fig.subplots_adjust(top=0.84, bottom=0.05, left=0.03, right=0.86, wspace=0.05)
for ax, y in zip(axes, (BASE_YEAR, END_YEAR)):
    draw_map(ax, y)
    ax.text(0.03, 0.97, str(y), transform=ax.transAxes, fontproperties=B.FP.get("mono_bold"), fontsize=15,
            color=B.MOSS, va="top", bbox=dict(boxstyle="round,pad=0.25", fc=B.PAPER, ec="none"))
fig.colorbar(plt.cm.ScalarMappable(norm=matplotlib.colors.Normalize(MAP_VMIN, MAP_VMAX), cmap=B.SEQ_CMAP),
             cax=fig.add_axes([0.88, 0.2, 0.02, 0.5]), label="10¹⁵ molec/cm²")
B.chart_title(axes[0], "South Gujarat from space", f"yearly mean NO₂ · {GRID_DEG}° grid · amber: {RADIUS_KM:.0f} km estate circles")
B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT}{TAG}")
fig.savefig(f"ft{FT_NUM}_maps.png", dpi=150); plt.show()

if GROUND_STATUS == "run":
    fig, ax = plt.subplots(figsize=(7, 6))
    for e in ESTATE_ORDER:
        d = GROUND_PAIRS[GROUND_PAIRS.estate == e]
        ax.scatter(d.sat, d.ground, s=26, color=SERIES_COL[e], label=e, alpha=0.8)
    ax.set_xlabel("satellite, 10¹⁵ molec/cm²"); ax.set_ylabel(f"ground monitor, monthly mean ({', '.join(GROUND_PAIRS.units.dropna().unique())})")
    ax.legend()
    B.chart_title(ax, "Satellite vs ground monitors", f"monthly means · r = {GROUND_R:.2f} (within-station)")
    B.source_note(fig, f"23N Field Test #{FT_NUM} · ground: OpenAQ (station licences printed above){TAG}")
    fig.tight_layout(); fig.savefig(f"ft{FT_NUM}_ground.png", dpi=150); plt.show()

## 6 · Interactive demo
Pick a year: you see the yearly NO₂ map of south Gujarat with the three estate circles, and the monthly series.
The cell doesn't block "Run all"; open the public `gradio.live` link it prints.

In [ ]:
# CELL: gradio
def year_figure(year):
    year = int(year)
    fig = plt.figure(figsize=(10, 9))
    ax1 = fig.add_axes([0.05, 0.40, 0.9, 0.55]); ax2 = fig.add_axes([0.1, 0.06, 0.85, 0.27])
    draw_map(ax1, year); ax1.set_title(f"Yearly mean NO₂, {year}", loc="left", fontsize=14)
    draw_monthly(ax2, upto=pd.Timestamp(year, 12, 31))
    txt = " · ".join(f"{e}: {RES[e]['yearly'].get(year, float('nan')):.2f}" for e in ESTATE_ORDER) + " (10^15 molec/cm²)"
    B.source_note(fig, f"{DATA_CREDIT}{TAG}")
    return fig, txt

try:
    import gradio as gr
    with gr.Blocks(title=f"23N Field Test #{FT_NUM}: industrial air from space") as demo:
        gr.Markdown(f"### 23N Field Test #{FT_NUM}: NO₂ over Vapi, Ankleshwar and Dahej from Sentinel-5P\n"
                    "Independent study on public data. Satellite columns are estimates, not ground concentrations.")
        dd_ = gr.Dropdown([str(y) for y in YEARS], value=str(END_YEAR), label="Year")
        out_plot, out_txt = gr.Plot(), gr.Textbox(label="Yearly mean over each estate")
        dd_.change(year_figure, dd_, [out_plot, out_txt])
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", type(e).__name__, e)

## 7 · Video (1080×1080, 30 fps, 58 s timeline saved at 1.5×, H.264) and cover (1080×1350)
Every map, line and number in the video comes from the run above.

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30

def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)

def _title(fig, title, sub):
    fig.text(0.055, 0.85, title, fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
    fig.text(0.055, 0.785, sub.upper(), fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")

MAP_SECONDS = 14.0
def draw_evidence(fig, t):
    # t runs 0..35 inside the evidence section (video seconds 5..40)
    B.header(fig, FT_NUM)
    if t < MAP_SECONDS:                                             # yearly maps, one year at a time
        y = YEARS[min(int(t / (MAP_SECONDS / len(YEARS))), len(YEARS) - 1)]
        _title(fig, "South Gujarat from space", f"Sentinel-5P · yearly mean NO₂ · {GRID_DEG}° grid")
        ax = fig.add_axes([0.055, 0.17, 0.75, 0.58]); draw_map(ax, y, big=True)
        cax = fig.add_axes([0.80, 0.25, 0.022, 0.42])
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=matplotlib.colors.Normalize(MAP_VMIN, MAP_VMAX), cmap=B.SEQ_CMAP), cax=cax)
        cb.outline.set_visible(False); cax.tick_params(labelsize=12)
        cax.set_title("10¹⁵\nmolec/cm²", fontsize=11, color=B.MUTED, loc="left")
        fig.text(0.945, 0.785, str(y), fontproperties=B.FP.get("mono_bold"), fontsize=22, color=B.MOSS, ha="right", va="top")
        B.caption(fig, "Darker = more NO₂ in the air column. Amber circles: the three estates, 10 km")
    elif t < 27:                                                    # monthly series drawing in
        _title(fig, "Month by month, seven years", f"NO₂ within {RADIUS_KM:.0f} km · shaded: monsoon")
        ax = fig.add_axes([0.12, 0.22, 0.83, 0.50])
        span = pd.Timestamp(DATA_END) - pd.Timestamp(DATA_START)
        draw_monthly(ax, upto=pd.Timestamp(DATA_START) + span * _ease(_prog(t, MAP_SECONDS + 0.3, 25)), big=True)
        B.caption(fig, "Winter highs, monsoon lows (clouds hide most monsoon days)")
    else:                                                           # change per estate
        _title(fig, f"{BASE_YEAR} to {END_YEAR}", "change in NO₂ · same months in both years")
        ax = fig.add_axes([0.13, 0.22, 0.82, 0.50])
        g = _ease(_prog(t, 27.1, 29.5))
        names = ESTATE_ORDER + ["Regional"]
        vals = [RES[e]["chg"] * 100 * g if RES[e]["chg"] == RES[e]["chg"] else 0 for e in names]
        ax.bar(range(len(names)), vals, color=[B.MOSS if v <= 0 else B.AMBER for v in vals], width=0.6)
        ax.axhline(0, color=B.INK, lw=1)
        finals = [abs(RES[e]["chg"] * 100) for e in names if RES[e]["chg"] == RES[e]["chg"]]
        lim = max([10] + [v * 1.35 for v in finals])
        ax.set_ylim(-lim, lim); ax.set_xticks(range(len(names)), [n if n != "Regional" else "Region" for n in names], fontsize=15)
        ax.tick_params(axis="y", labelsize=13); ax.set_ylabel("% change", fontsize=14)
        if g >= 1:
            for i, e in enumerate(names):
                v = RES[e]["chg"]
                if v == v:
                    ax.text(i, v * 100 + (lim * 0.06 if v >= 0 else -lim * 0.06), pct(v), ha="center",
                            va="bottom" if v >= 0 else "top", fontproperties=B.FP.get("mono_bold"), fontsize=16, color=B.INK)
        B.caption(fig, GROUND_LINE)

def render_video(path=VIDEO_PATH):
    v = B.Video(path, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "Is the air over Gujarat's industrial estates getting cleaner?",
                f"Sentinel-5P NO₂ · Vapi, Ankleshwar, Dahej · {DATA_START[:4]}-{DATA_END[:4]}"); v.hold(5)
    n = int(35 * FPS); key0, buf = None, None
    for f in range(n):
        t = f / FPS
        if t < MAP_SECONDS: key = ("m", min(int(t / (MAP_SECONDS / len(YEARS))), len(YEARS) - 1))
        elif t < 27:        key = ("s", min(round(t * 10) / 10, 25.2))
        else:               key = ("b", min(round(t * 10) / 10, 29.6))
        if key != key0:
            v.clear(); draw_evidence(v.fig, t); buf = v.grab(); key0 = key
        v.proc.stdin.write(buf); v.frames += 1
    v.clear(); B.headline_card(v.fig, FT_NUM, BIG, BIG_LINE,
                               f"Sentinel-5P TROPOMI tropospheric NO₂, qa >= {QA_MIN}, {RADIUS_KM:.0f} km circles, "
                               f"{BASE_YEAR} vs {END_YEAR} on matched months. Estimate.")
    v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, DATA_CREDIT); v.hold(6)
    v.close()
    print("video frames:", v.frames, f"({v.frames / FPS:.1f}s timeline)")

def render_cover(path=COVER_PATH):
    def panel(ax):
        draw_map(ax, END_YEAR, big=True)
    B.cover(path, FT_NUM, "Industrial air, measured from space", f"Vapi · Ankleshwar · Dahej · Sentinel-5P NO₂ · {END_YEAR}",
            BIG, f"NO₂ over {TOP}, {BASE_YEAR} to {END_YEAR} (estimate)", image=panel, data_credit="Copernicus Sentinel data")

In [ ]:
# CELL: video_render
render_video()
render_cover()
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 8 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """Is the air over Gujarat's industrial estates getting cleaner? We measured NO₂ over Vapi, Ankleshwar and Dahej from space, 2019 to 2025.

23N Field Test #08: {headline}

{cepi_fact}.

How we tested it:
• Copernicus Sentinel-5P tropospheric NO₂, {n_over} overpasses, {period}, pixels within {radius} km of each estate, qa ≥ {qa}
• Years compared on the same months only; a south Gujarat regional level alongside
• {ground_line}

Result: {result_line}

Limits: a satellite column is not a ground concentration; one early-afternoon overpass a day; clouds hide most of the monsoon. An estimate.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """NO₂ over Gujarat's industrial estates, {period}, from Sentinel-5P: {x_changes}. Region {chg_region}. 23N Field Test #08. Notebook below.

---
Thread (optional):
1/ Data: Copernicus Sentinel-5P TROPOMI L2 NO₂ via Microsoft Planetary Computer, no login. Pixels within {radius} km of each estate, qa ≥ {qa}.
2/ Method: matched months only, so a cloudier year can't fake a trend. {ground_short}
3/ Limits: a column above the estate, not what people breathe; afternoon overpass only. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
RESULT_LINE = "; ".join(f"{e} {RES[e]['v0']:.1f} → {RES[e]['v1']:.1f} ({pct(CHG[e])})" for e in ESTATE_ORDER) + " ×10¹⁵ molec/cm²."
POST_VALUES = dict(headline=HEADLINE, cepi_fact=CEPI_FACT, cepi_source=CEPI_SOURCE, n_over=f"{PIX['orbit'].nunique():,}",
                   period=f"{BASE_YEAR}–{END_YEAR}", radius=f"{RADIUS_KM:.0f}", qa=QA_MIN, ground_line=GROUND_LINE,
                   result_line=RESULT_LINE, chg_region=pct(rc),
                   x_changes=", ".join(f"{e} {pct(CHG[e])}" for e in ESTATE_ORDER),
                   ground_short=(f"Ground monitors: r = {GROUND_R:.2f}." if GROUND_STATUS == "run" else "Ground check not run."),
                   signature=B.post_signature(FT_NUM) + " #AirQuality #EarthObservation #Gujarat")
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): trim before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): trim before posting")

## 9 · Limitations (read before quoting the number)

- **A column, not a concentration.** TROPOMI measures all the NO₂ between the ground and roughly the top of the
  boundary layer, averaged over a ~20 km² pixel. It is not what a person at the gate breathes, and it can't be
  compared with the national ambient standard (µg/m³). It is good for **changes and comparisons**.
- **One look a day, early afternoon.** The overpass is at about 13:30 local time. Night-time emissions and morning
  peaks are not seen.
- **Clouds.** With `qa_value ≥ 0.75`, most monsoon days are gone. Matched months stop that from faking a trend,
  but the yearly level is mostly a dry-season level.
- **Weather and neighbours.** Wind, boundary-layer height and the sun change the column from year to year. The
  circles are not only the estates: Vapi's includes the town, Ankleshwar's reaches Bharuch city across the
  Narmada, Dahej's includes the port. Surat sits in the regional box.
- **Processing versions.** Retrieval versions change over time. The notebook prints the version mix and stops the
  headline if the base and end years use different major versions.
- **Sampling.** `DAYS_PER_MONTH` days are read per month, not every day, to fit a free notebook. More days → less
  noise.
- **Ground check.** Optional and correlational: monthly ground means against the satellite at the station's
  location. Monitors near these estates started in recent years and have gaps.
- **Not a statement about any company, estate authority or regulator.** The estates are used only as locations.
- **Licences.** Copernicus Sentinel data terms: free, full and open; attribution "Contains modified Copernicus
  Sentinel data 2019–2025". Access via Microsoft Planetary Computer (no account). Ground data via OpenAQ: each
  station's licence is printed with the data. Code in this repo: see LICENSE.